# Semana 22 — Preference Alignment: DPO do Zero
**Fase 4: Parameter-Efficient Fine-Tuning (PEFT)** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/04-Phase-4-Fine-Tuning/notebooks/semana_22_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-22-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/04-Phase-4-Fine-Tuning/docs/Semana-22-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_22_gabarito.ipynb`.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

In [ ]:
!pip -q install peft

In [ ]:
import torch, torch.nn.functional as F, random, math, pandas as pd, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0); print(device)
MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
ALVOS = ["q_proj", "k_proj", "v_proj", "o_proj"]

## Passo 1 — Dados de preferência
Mesma informação, dois tons. A empresa prefere o tom formal. A única diferença sistemática entre escolhida e rejeitada é o tom.

In [ ]:
fatos = [("Como redefino minha senha?", "a redefinição é feita no portal de identidades, opção Esqueci minha senha"),
         ("Onde vejo meu holerite?", "o holerite fica disponível no portal RH, menu Pagamentos"),
         ("Como peço um notebook novo?", "o pedido é aberto no service desk, categoria Equipamentos"),
         ("Qual o prazo do reembolso?", "o reembolso é pago em até 10 dias úteis após o envio dos recibos"),
         ("Como acesso a VPN?", "o acesso exige o cliente GlobalConnect e o certificado pessoal"),
         ("Posso trabalhar de casa na sexta?", "o trabalho remoto às sextas depende de acordo com o gestor"),
         ("Como marco férias?", "as férias são solicitadas no portal RH com 45 dias de antecedência"),
         ("Quem aprova uma licença de software?", "a aprovação cabe ao gestor e ao time de compras"),
         ("Onde reporto um incidente de segurança?", "o incidente deve ser reportado ao SOC em até 2 horas"),
         ("Como incluo um dependente no plano de saúde?", "a inclusão é feita no portal de benefícios com a certidão do dependente"),
         ("Quando cai o vale-refeição?", "o crédito ocorre no último dia útil do mês"),
         ("Como reservo uma sala de reunião?", "a reserva é feita pelo calendário corporativo, na aba Salas"),
         ("Onde encontro o organograma?", "o organograma está na intranet, seção Institucional"),
         ("Como peço um cartão corporativo?", "o cartão é solicitado ao financeiro pelo formulário de despesas"),
         ("Qual o canal para dúvidas de TI?", "as dúvidas de TI são atendidas pelo service desk, ramal 4000"),
         ("Como atualizo meus dados bancários?", "a atualização é feita no portal RH, menu Dados pessoais")]
formal = lambda r: f"Prezado(a) colaborador(a), informamos que {r}. Permanecemos à disposição."
informal = lambda r: f"E aí, beleza? Então, {r}, tá? Qualquer coisa grita aí, valeu!"
pares = [{"prompt": [{"role": "user", "content": q}], "chosen": formal(r), "rejected": informal(r)} for q, r in fatos]
random.Random(0).shuffle(pares); treino, reservado = pares[:12], pares[12:]
print(treino[0])

## Passo 2 — Modelo e adapter

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO)
modelo = get_peft_model(AutoModelForCausalLM.from_pretrained(MODELO).to(device),
                        LoraConfig(r=8, lora_alpha=16, target_modules=ALVOS, task_type="CAUSAL_LM")).eval()
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

def gerar(mensagens, max_new_tokens=50):
    ent = tok(tok.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True), return_tensors="pt", add_special_tokens=False).to(device)
    with torch.no_grad(): out = modelo.generate(**ent, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=PAD)
    return tok.decode(out[0, ent["input_ids"].shape[1]:], skip_special_tokens=True).strip()

teste_q = [{"role": "user", "content": "Como solicito um crachá novo?"}]
antes = gerar(teste_q); print("ANTES:", antes)

## Passo 3 — Log-probabilidade de uma resposta
`logprob(mensagens, resposta)` = soma de `log P(token)` de cada token da resposta (incluindo o token de fim), dado o prompt.

Dica: se a sequência é `prompt + resposta`, os logits da posição `t` preveem o token `t + 1`. Os logits que preveem os tokens da resposta estão nas posições `len(prompt) - 1` até a penúltima.

In [ ]:
def logprob(mensagens, resposta):
    prompt = tok.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    ids_p = tok(prompt, add_special_tokens=False)["input_ids"]
    ids_r = tok(resposta + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = torch.tensor([ids_p + ids_r], device=device)
    # [SEU CÓDIGO AQUI] selecione os logits que preveem a resposta, aplique log_softmax e some os log-probs dos tokens corretos

with torch.no_grad():
    p = treino[0]; print("log P(formal):", round(logprob(p["prompt"], p["chosen"]).item(), 1), "| log P(informal):", round(logprob(p["prompt"], p["rejected"]).item(), 1))

## Passo 4 — Log-probabilidades da referência
O modelo de referência é o modelo **antes** do DPO. Calcule uma vez, sem gradiente, com o adapter desligado.

In [ ]:
with torch.no_grad(), modelo.disable_adapter():
    for p in pares:
        # [SEU CÓDIGO AQUI] guarde ref_c e ref_r em cada par

def taxa_preferencia(conj):
    with torch.no_grad():
        return sum(logprob(p["prompt"], p["chosen"]).item() > logprob(p["prompt"], p["rejected"]).item() for p in conj) / len(conj)
print("taxa de preferência pelo formal antes do DPO (reservado):", taxa_preferencia(reservado))

## Passo 5 — A loss do DPO
`L = -log σ(β · [(π_c − ref_c) − (π_r − ref_r)])`. Use `F.logsigmoid`.

In [ ]:
def perda_dpo(pi_c, pi_r, ref_c, ref_r, beta=0.1):
    # [SEU CÓDIGO AQUI] implemente a fórmula

z = torch.tensor(0.0)
assert abs(perda_dpo(z, z, 0.0, 0.0).item() - math.log(2)) < 1e-6
with torch.no_grad():
    p = treino[0]; print("perda inicial:", round(perda_dpo(logprob(p["prompt"], p["chosen"]), logprob(p["prompt"], p["rejected"]), p["ref_c"], p["ref_r"]).item(), 4), "(esperado ≈ 0.6931)")

## Passo 6 — Treino
Para cada par: calcule as duas log-probabilidades **com gradiente**, a loss e atualize o adapter.

In [ ]:
treinaveis = [p for p in modelo.parameters() if p.requires_grad]
opt = torch.optim.AdamW(treinaveis, lr=2e-4); hist = []; rnd = random.Random(0); modelo.train()
for epoca in range(6):
    rnd.shuffle(treino)
    for p in treino:
        # [SEU CÓDIGO AQUI] loss, backward, clip e passo do otimizador
        hist.append(loss.item())
    print(f"época {epoca} | perda média {sum(hist[-len(treino):]) / len(treino):.4f}")
modelo.eval(); plt.plot(hist); plt.axhline(math.log(2), ls="--", c="gray"); plt.ylabel("perda DPO"); plt.show()

## Passo 7 — O comportamento mudou?

In [ ]:
print("taxa de preferência pelo formal depois do DPO (reservado):", taxa_preferencia(reservado))
print("\nANTES :", antes); print("DEPOIS:", gerar(teste_q))
for p in reservado[:2]: print("\nQ:", p["prompt"][0]["content"], "\nR:", gerar(p["prompt"]))

## Passo 8 — Margem em relação à referência
Para cada par reservado, calcule o quanto a política aumentou a log-probabilidade da escolhida e o quanto mudou a da rejeitada, ambas em relação à referência.

In [ ]:
linhas = []
with torch.no_grad():
    for p in reservado:
        # [SEU CÓDIGO AQUI] d_c e d_r: variação da log-prob da escolhida e da rejeitada em relação à referência
        linhas.append({"pergunta": p["prompt"][0]["content"], "Δ escolhida": d_c, "Δ rejeitada": d_r, "margem": d_c - d_r})
pd.DataFrame(linhas).round(2)

O DPO só exige que a **margem** cresça. Observe: a log-probabilidade da escolhida subiu, a da rejeitada caiu, ou as duas coisas?

## Passo 9 — Drift em tarefa não relacionada

In [ ]:
geral = [{"role": "user", "content": "Quanto é 12 vezes 8? Responda só o número."}]
with modelo.disable_adapter(): print("REFERÊNCIA:", gerar(geral, 12))
print("COM DPO   :", gerar(geral, 12))

**Reflexão:** o modelo passou a responder perguntas de matemática com "Prezado(a) colaborador(a)"? Se sim, é drift de comportamento. O que você mudaria: `β`, número de epochs, ou o dataset (incluindo pares em que a resposta curta e direta é a escolhida)?